## Prepare ADRD dataset

In [2]:
## Load packages ----
import numpy as np
import pandas as pd
import sshtunnel
import psycopg2 as pg
import os

import json
import sys

import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
## read zip to county crosswalk ----
zip_to_county = pd.read_csv('../data/input/remote/zip_county_2010.csv')
zip_to_county = zip_to_county[['ZIP', 'COUNTY']]
zip_to_county = zip_to_county.rename(columns = {'ZIP':'zip', 'COUNTY':'county'})
zip_w = zip_to_county.groupby(['zip'])['county'].count().reset_index()
zip_w = zip_w.rename(columns = {'county':'w'})
zip_w['w'] = 1 / zip_w.w
zip_to_county = zip_to_county.merge(zip_w)
zip_to_county.w.describe()

count    46875.000000
mean         0.775168
std          0.278345
min          0.166667
25%          0.500000
50%          1.000000
75%          1.000000
max          1.000000
Name: w, dtype: float64

In [5]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'{os.environ["MY_NSAPH_SSH_USERNAME"]}',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'{os.environ["MY_NSAPH_SSH_PASSWORD"]}', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()

## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

In [6]:
## define functions ----
def get_outcomes(read_path):
    """ Get and return ICD codes """""
    f = open(read_path)
    res_dict = json.load(f)
    f.close()
    res_dict = json.loads(res_dict[0])
    return res_dict

def get_outcomes_set(outcome=None, year=None):
    """ Uses ICD9 for years prior 2015 and ICD10 otherwise """
    if year < 2015:
        outcomes_set = outcomes[outcome]["icd9"]
    elif year > 2015:
        outcomes_set = outcomes[outcome]["icd10"]
    else:
        outcomes_set = outcomes[outcome]["icd10"] + \
                       outcomes[outcome]["icd9"]
    return set(outcomes_set)

def get_outcome_in_diagnoses(outcomes_set=None, diagnoses=None):
    return any(o_ in outcomes_set for o_ in diagnoses)

## Beneficiary counts

In [6]:
## year range of interest ----
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015) # not available in DB as of Nov 2022
years_.remove(2006) # not available in DB as of Nov 2022

## obtain beneficiary counts per zipcode ----

bene_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT 
        zip,
        year,
        race, 
        sex,
        case 
            when age < 65 then '<65'
            when age >= 65 and age < 75 then '[65,75)'
            when age >= 75 and age < 85 then '[75,85)'
            when age >= 85 then '>85'
        end age_grp,
        count(*) as n_enrollees
    FROM 
        medicare.enrollments
        LEFT JOIN medicare.beneficiaries ON medicare.enrollments.bene_id = medicare.beneficiaries.bene_id
    WHERE 
        year in ('{y_}') AND
        state = 'NC' AND 
        race in ('1', '2') AND
        sex in ('1', '2')
    GROUP BY 
        age_grp, 
        year, 
        zip, 
        race, 
        sex
    ;
    """
    ## Request query ----
    %time b = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    bene_zip_list.append(b)

2000
CPU times: user 355 ms, sys: 72.4 ms, total: 427 ms
Wall time: 1min 43s
2001
CPU times: user 115 ms, sys: 16.1 ms, total: 132 ms
Wall time: 17.4 s
2002
CPU times: user 107 ms, sys: 25.7 ms, total: 132 ms
Wall time: 17.9 s
2003
CPU times: user 104 ms, sys: 13.7 ms, total: 118 ms
Wall time: 20.5 s
2004
CPU times: user 98.4 ms, sys: 17.3 ms, total: 116 ms
Wall time: 19.4 s
2005
CPU times: user 102 ms, sys: 15.5 ms, total: 117 ms
Wall time: 19.9 s
2007
CPU times: user 129 ms, sys: 18.7 ms, total: 148 ms
Wall time: 1min 28s
2008
CPU times: user 88.6 ms, sys: 32.3 ms, total: 121 ms
Wall time: 19.9 s
2009
CPU times: user 103 ms, sys: 15.5 ms, total: 118 ms
Wall time: 18.1 s
2010
CPU times: user 98.8 ms, sys: 19.1 ms, total: 118 ms
Wall time: 17.6 s
2011
CPU times: user 101 ms, sys: 18.3 ms, total: 120 ms
Wall time: 20.5 s
2012
CPU times: user 106 ms, sys: 27.7 ms, total: 133 ms
Wall time: 45 s
2013
CPU times: user 105 ms, sys: 17.4 ms, total: 122 ms
Wall time: 20.2 s
2014
CPU times: user

In [7]:
## crosswalk to counties ----
bene_zip_df = pd.concat(bene_zip_list)
bene_county_df = bene_zip_df.merge(zip_to_county)
bene_county_df['n_enrollees'] = bene_county_df.n_enrollees * bene_county_df.w
bene_county_df = bene_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_enrollees'].sum().reset_index()

In [8]:
## total number of enrollees in zipcodes ----
bene_zip_df.n_enrollees.sum()

25491220

In [9]:
## total number of enrollees in counties ----
bene_county_df.n_enrollees.sum()

24320746.0

## Admission counts

In [7]:
## year range of interest ----
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015) # not available in DB as of Nov 2022
years_.remove(2006) # not available in DB as of Nov 2022

## obtain adrd counts per zipcode ----
adm_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT
        bene.bene_id,
        diagnoses,
        zip,
        year,
        race,
        sex, 
        EXTRACT(YEAR FROM dob) as yob_
    FROM 
        medicare.beneficiaries as bene
    RIGHT JOIN (
        SELECT 
            bene_id, 
            diagnoses, 
            zip, 
            year
        FROM 
            medicare.admissions as adm
        WHERE
            year in ('{y_}') AND
            state = 'NC'
    ) as adm
    ON bene.bene_id = adm.bene_id
    WHERE
      race in ('1', '2') AND
      sex in ('1', '2')
    ;
    """
    ## Request query ----
    %time a = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    adm_zip_list.append(a)

2000
CPU times: user 5.39 s, sys: 1.15 s, total: 6.54 s
Wall time: 24.5 s
2001
CPU times: user 5.18 s, sys: 1.02 s, total: 6.2 s
Wall time: 18.2 s
2002
CPU times: user 5.18 s, sys: 1.02 s, total: 6.2 s
Wall time: 18.7 s
2003
CPU times: user 6.33 s, sys: 1.28 s, total: 7.61 s
Wall time: 19.7 s
2004
CPU times: user 6.81 s, sys: 1.14 s, total: 7.95 s
Wall time: 17.9 s
2005
CPU times: user 7.35 s, sys: 1.09 s, total: 8.44 s
Wall time: 17.2 s
2007
CPU times: user 6.55 s, sys: 1.57 s, total: 8.12 s
Wall time: 18.1 s
2008
CPU times: user 6.47 s, sys: 1.67 s, total: 8.14 s
Wall time: 16.1 s
2009
CPU times: user 7.07 s, sys: 1.56 s, total: 8.63 s
Wall time: 15.8 s
2010
CPU times: user 9.02 s, sys: 1.84 s, total: 10.9 s
Wall time: 15.7 s
2011
CPU times: user 10.5 s, sys: 3.01 s, total: 13.5 s
Wall time: 15 s
2012
CPU times: user 13.7 s, sys: 3.22 s, total: 16.9 s
Wall time: 18.3 s
2013
CPU times: user 14.3 s, sys: 3.35 s, total: 17.7 s
Wall time: 18.5 s
2014
CPU times: user 11.7 s, sys: 3.2 s, t

In [10]:
adm_zip_df = pd.concat(adm_zip_list)
adm_zip_df['age'] = adm_zip_df.year - adm_zip_df.yob_
adm_zip_df['age_grp'] = pd.cut(x=adm_zip_df['age'], 
                               bins=[min(adm_zip_df.age), 65, 75, 85, max(adm_zip_df.age)],
                               labels=['<65', '[65,75)', '[75,85)', '>85'])

## read outcomes ----
read_path = '../data/input/remote/icd_codes.json'
outcomes = get_outcomes(read_path)

## find diagnoses ----
for outcome in ['adrd']:
    adm_zip_df[outcome] = [get_outcome_in_diagnoses(get_outcomes_set(outcome, y_), d_[:1]) for y_, d_ in zip(adm_zip_df.year, adm_zip_df.diagnoses)]

In [12]:
adm_zip_df[['year', 'adrd']].groupby(['year']).sum()

,adrd
year,
2000,2749
2001,2777
2002,3040
2003,4805
2004,4815
2005,4575
2007,3400
2008,3472
2009,3124


In [13]:
keep = adm_zip_df[['adrd']].any(axis=1)
adm_zip_df = adm_zip_df[keep]
adm_zip_df = adm_zip_df.drop(columns='adrd')
adm_zip_df = adm_zip_df.groupby(['year', 'zip', 'race', 'sex', 'age_grp'])['bene_id'].count().reset_index()
adm_zip_df = adm_zip_df.rename(columns = {'bene_id':'n_adrd'})

In [14]:
## crosswalk to counties ----
adm_county_df = adm_zip_df.merge(zip_to_county)
adm_county_df['n_adrd'] = adm_county_df.n_adrd * adm_county_df.w
adm_county_df = adm_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_adrd'].sum().reset_index()

In [15]:
## total number of adrd admissions in zipcodes ----
adm_zip_df.n_adrd.sum()

36092

In [16]:
## total number of adrd admissions in counties ----
adm_county_df.n_adrd.sum()

34900.0

## Adrd counts

In [14]:
## obtain rows for all combinations of county, year, race, sex and age_grp ----
## merge with enrollee and adrd counts
## there may be missing counts for a given combination
county_ = sorted(bene_county_df.county.unique())
year_ = sorted(bene_county_df.year.unique())
race_ = sorted(bene_county_df.race.unique())
sex_ = sorted(bene_county_df.sex.unique())
age_grp_ = sorted(bene_county_df.age_grp.unique())

In [16]:
adrd_county_df = pd.DataFrame({'county':county_}).merge(pd.DataFrame({'year':year_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'race':race_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'sex':sex_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'age_grp':age_grp_}), how = 'cross')

In [17]:
adrd_county_df['state'] = [str(x)[0:2] for x in adrd_county_df.county]
adrd_county_df = adrd_county_df[adrd_county_df.state == '37']

In [19]:
adrd_county_df = adrd_county_df.merge(bene_county_df, how = 'left')
adrd_county_df = adrd_county_df.merge(adm_county_df, how = 'left')

In [15]:
## total number of counties in bene_county_df (resulting from crosswalk)----
len(county_)

1125

In [18]:
## total number of counties in adrd_county_df (after filtering NC) ----
len(adrd_county_df.county.unique())

100

In [20]:
## total number of enrollees in adrd_county_df ----
adrd_county_df.n_enrollees.sum()

24311184.0

In [21]:
## total number of adrd admissions in adrd_county_df ----
adrd_county_df.n_adrd.sum()

34890.0

In [22]:
## percentage of county-race-sex-age_grp combinations with missing enrollees (all years) ----
adrd_county_df.n_enrollees.isnull().mean()

0.010073529411764705

In [23]:
## percentage of county-race-sex-age_grp combinations with missing adrd hospitalizations (all years) ----
adrd_county_df.n_adrd.isnull().mean()

0.4117647058823529

In [24]:
## save adrd_county_df
adrd_county_df.to_csv("../data/input/adrd_county_df.csv", index=False)